# Segger segmentation for decoded ISS transcripts

Segger has a Python/CUDA/RAPIDS stack that must remain separate from the Python 3.10 ISS environment. Keep this notebook on the `ISS_postprocessing` kernel and create the isolated runtime once from the repository root:

```bash
conda env create -f ISS_postprocessing/environments/segger.yml
```

The environment follows Segger v0.3.0's upstream CUDA 12.1/RAPIDS 24.10 stack and is intended for Linux with an NVIDIA GPU. This notebook launches the Segger CLI and its format bridge through `conda run`, then writes the standard sparse `.npz` mask.

In [ ]:
from pathlib import Path
import ISS_postprocessing.segmentation as SEG

input_dir = Path('/path/to/regions')
region = 'R1'
pixel_size_um = 0.325
transcripts_file = input_dir / region / 'decoding' / '2_decoded' / f'{region}_decoded.csv'
dapi_file = input_dir / region / 'preprocessing' / 'Cycle1' / '3_stitched' / 'Cycle1_ch4.tif'
initial_mask = None

In [ ]:
labels, labels_coo = SEG.segger_segmentation(
    transcripts=transcripts_file,
    image=dapi_file,
    initial_mask=initial_mask,
    region=region,
    input_dir=input_dir,
    pixel_size_um=pixel_size_um,
    segger_command=['conda', 'run', '-n', 'segger', 'segger'],
    segger_python=['conda', 'run', '-n', 'segger', 'python'],
    extra_args=['--n-epochs', '20'],
)
labels.shape, int(labels.max()), labels_coo.nnz